In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import glob
from xgrads import open_CtlDataset
from pathlib import Path
import netCDF4

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib.colors as mcolors

import ipywidgets as widgets
from IPython.display import display, clear_output

ncl_cmap = LinearSegmentedColormap.from_list(
    "BlueWhiteOrangeRed",
    ["#2166ac", "#67a9cf", "#ffffff", "#fdae61", "#b2182b"],
    N=256
)


plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_colwidth", 160)

print("Python OK")

: 

In [ ]:
BASE_DIR = Path.cwd()

NEW_DIR = Path.cwd() / ".." / ".." / ".." / "SPEEDY_access" / "output" / "exp_102"
OLD_DIR = Path.cwd() / ".." / ".." / ".." / "SPEEDY_access" / "output" / "exp_101"
print("NEW_DIR:", NEW_DIR, NEW_DIR.exists())
print("OLD_DIR:", OLD_DIR, OLD_DIR.exists())

# print("\nNew files:")
# if NEW_DIR.exists():
#     for p in sorted(NEW_DIR.glob("*.grd")):
#         print(f"{p.name:25s} {p.stat().st_size / 1024**2:.2f} MB")
# else:
#     print("NEW_DIR does not exist")

# print("\nOld files:")
# if OLD_DIR.exists():
#     for p in sorted(OLD_DIR.glob("*.grd")):
#          print(f"{p.name:25s} {p.stat().st_size / 1024**2:.2f} MB")
# else:
#     print("OLD_DIR does not exist")   

SPEEDY_VARIABLE = "Q0"
ACCESS_VARIABLE = "huss"   

# Output directory
TAS_OUT_DIR = (Path.cwd() / ".." / ".." / "access_forcing").resolve()
TAS_OUT_DIR.mkdir(parents=True, exist_ok=True)
WRITE_ONE_FILE_PER_YEAR = True
# Keep the SPEEDY grid untouched until the JRA-55 metadata/grid are inspected.
SHIFT_LONGITUDE_TO_MINUS180_180 = False
SORT_LATITUDE_NORTH_TO_SOUTH = False
print("Output directory:", TAS_OUT_DIR)


: 

In [ ]:
for ctl in Path(NEW_DIR).glob("attm102.ctl"):
    print("=" * 80)
    print(ctl.name)

    ds = open_CtlDataset(str(ctl))
ds

: 

In [ ]:
# Inspect source 


if SPEEDY_VARIABLE not in ds:
    raise KeyError(
        f"{SPEEDY_VARIABLE!r} is absent from the SPEEDY dataset. "
        f"Available variables: {list(ds.data_vars)}"
    )

st = ds[SPEEDY_VARIABLE]

print(st)
print("\nDimensions:", st.dims)
print("Shape:", st.shape)
print("Dtype:", st.dtype)
print("Attributes:", st.attrs)

dt_hours = np.diff(ds.time.values) / np.timedelta64(1, "h")
print("\nUnique output intervals [hours]:", np.unique(dt_hours))

print(
    "\nQ0 range [g/kg]:",
    float(st.min().compute()),
    "to",
    float(st.max().compute()),
)

print(
    "Q0  global mean [g/kg]:",
    float(st.mean(skipna=True).compute()),
)

print("\nUnique output intervals [hours]:", np.unique(dt_hours))
print("First time:", ds.time.values[0])
print("Last time:", ds.time.values[-1])

: 

In [ ]:
# Build huss

huss = (ds[SPEEDY_VARIABLE] / 1000.0).rename(ACCESS_VARIABLE).astype("float32")

if SHIFT_LONGITUDE_TO_MINUS180_180:
    huss = huss.assign_coords(lon=((huss.lon + 180.0) % 360.0) - 180.0).sortby("lon")

if SORT_LATITUDE_NORTH_TO_SOUTH:
    huss = huss.sortby("lat", ascending=False)

huss.attrs = {
    "standard_name": "specific_humidity",
    "long_name": "Near-Surface Specific Humidity",
    "comment": "Near-surface specific humidity from SPEEDY Q0",
    "units": "1",
    "cell_methods": "area: mean time: point",
    "source_variable": "Q0",
    "source_model": "SPEEDY",
    "mapping_note": "SPEEDY Q0 converted from g kg-1 to kg kg-1 -> ACCESS-OM2 huss",
}

huss_ds = huss.to_dataset()

# JRA55-do 3hrPt convention: point time with ±1.5 h bounds
time = huss_ds.time
huss_ds["time_bnds"] = xr.DataArray(
    np.stack([(time - np.timedelta64(90, "m")).values,
              (time + np.timedelta64(90, "m")).values], axis=1),
    dims=("time", "bnds"), coords={"time": time, "bnds": [0, 1]}
)

huss_ds["lat"].attrs.update({"standard_name": "latitude", "long_name": "Latitude", "units": "degrees_north", "axis": "Y"})
huss_ds["lon"].attrs.update({"standard_name": "longitude", "long_name": "Longitude", "units": "degrees_east", "axis": "X"})
huss_ds["time"].attrs.update({"standard_name": "time", "long_name": "time", "axis": "T", "bounds": "time_bnds"})

huss_ds.attrs = {
    "Conventions": "CF-1.7",
    "title": "SPEEDY forcing for ACCESS-OM2",
    "source": "SPEEDY model output",
    "frequency": "3hrPt",
    "history": "Created from SPEEDY Q0 and exported as huss",
    "comment": "3-hourly point near-surface specific humidity forcing following JRA55-do temporal convention.",
}

huss_ds

: 

In [ ]:
# =========================
# Basic validation

required_dims = ("time", "lat", "lon")

if huss.dims != required_dims:
    raise ValueError(f"Expected huss dimensions {required_dims}, got {huss.dims}")

if huss.attrs.get("units") != "1":
    raise ValueError("huss must be stored as kg kg-1 (dimensionless)")

if huss.attrs.get("cell_methods") != "area: mean time: point":
    raise ValueError(f"Unexpected cell_methods: {huss.attrs.get('cell_methods')}")

if huss_ds.attrs.get("frequency") != "3hrPt":
    raise ValueError(f"Unexpected frequency: {huss_ds.attrs.get('frequency')}")

if not np.issubdtype(huss.dtype, np.floating):
    raise TypeError(f"huss must be floating point, got {huss.dtype}")

# Missing/invalid values
if not bool(np.isfinite(huss).all().compute()):
    raise ValueError("huss contains NaN or infinite values")

undef = ds.attrs.get("undef", 9.999e19) / 1000.0
invalid_count = int((np.abs(huss) >= abs(undef) * 0.9).sum().compute())
if invalid_count:
    raise ValueError(f"Found {invalid_count} values close to converted SPEEDY undef={undef}")

# Time axis: JRA55-do 3hrPt convention
dt_hours = np.diff(huss.time.values) / np.timedelta64(1, "h")
if not np.all(dt_hours == 3):
    raise ValueError(f"Expected 3-hourly time axis, got {np.unique(dt_hours)} h")

if "time_bnds" not in huss_ds:
    raise ValueError("time_bnds is missing")

bounds = huss_ds["time_bnds"]
width_hours = (bounds[:, 1] - bounds[:, 0]).values / np.timedelta64(1, "h")
midpoints = bounds[:, 0].values + (bounds[:, 1].values - bounds[:, 0].values) / 2

if not np.all(width_hours == 3):
    raise ValueError("time_bnds must be exactly 3 hours wide")

if not np.array_equal(midpoints, huss.time.values):
    raise ValueError("huss time must be the midpoint of time_bnds")

# Broad physical sanity check
huss_min = float(huss.min().compute())
huss_max = float(huss.max().compute())

if huss_min < 0.0:
    print(f"WARNING: negative minimum huss={huss_min:.6f}")

if huss_max > 0.05:
    print(f"WARNING: unusually high maximum huss={huss_max:.6f}")

print("Validation passed")
print(f"huss range: {huss_min:.6f} to {huss_max:.6f} kg kg-1")
print("time:", huss.time.values[0], "to", huss.time.values[-1])
print("intervals [h]:", np.unique(dt_hours))
print("grid:", huss.sizes["lat"], "x", huss.sizes["lon"])

: 

In [ ]:
# 5. NetCDF encoding

field_encoding = {
    "dtype": "float32", "zlib": True, "complevel": 4, "shuffle": True,
    "_FillValue": np.float32(1.0e20),
    "chunksizes": (1, huss.sizes["lat"], huss.sizes["lon"]),
}

encoding = {
    ACCESS_VARIABLE: field_encoding,
    "time": {"dtype": "float64", "units": "days since 1900-01-01 00:00:00", "calendar": "gregorian", "_FillValue": None},
    "time_bnds": {"dtype": "float64", "units": "days since 1900-01-01 00:00:00", "calendar": "gregorian", "_FillValue": None},
    "lat": {"dtype": "float64", "_FillValue": None},
    "lon": {"dtype": "float64", "_FillValue": None},
}

encoding

: 

In [ ]:
# =========================
# Write NetCDF files

written_files = []

if WRITE_ONE_FILE_PER_YEAR:
    years = np.unique(huss_ds.time.dt.year.values)

    for year in years:
        yearly = huss_ds.sel(time=str(int(year)))

        if yearly.sizes["time"] != 2920:
            raise ValueError(f"{year}: expected 2920 3-hourly records, got {yearly.sizes['time']}")

        output_file = TAS_OUT_DIR / f"huss_SPEEDY_{int(year)}.nc"

        yearly.to_netcdf(
            output_file,
            mode="w",
            format="NETCDF4",
            engine="netcdf4",
            unlimited_dims=["time"],
            encoding=encoding,
        )

        written_files.append(output_file)
        print(f"Wrote {output_file.name}: {yearly.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

else:
    output_file = TAS_OUT_DIR / "huss_SPEEDY_all_years.nc"

    huss_ds.to_netcdf(
        output_file,
        mode="w",
        format="NETCDF4",
        engine="netcdf4",
        unlimited_dims=["time"],
        encoding=encoding,
    )

    written_files.append(output_file)
    print(f"Wrote {output_file.name}: {huss_ds.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

written_files

: 

In [ ]:
# Reopen and verify output

if not written_files:
    raise RuntimeError("No NetCDF files were written")

check_file = written_files[0]

with xr.open_dataset(check_file, decode_times=True) as check:
    print(check)
    print("\nVariable attributes:", check[ACCESS_VARIABLE].attrs)
    print("\nEncoding:", check[ACCESS_VARIABLE].encoding)
    print("\nTime:", check.time.values[0], "to", check.time.values[-1])

    if check.sizes["time"] != 2920:
        raise ValueError(f"Expected 2920 records, got {check.sizes['time']}")

    if check.attrs.get("frequency") != "3hrPt":
        raise ValueError(f"Unexpected frequency: {check.attrs.get('frequency')}")

    if "time_bnds" not in check:
        raise ValueError("time_bnds is missing")

    dt_hours = np.diff(check.time.values) / np.timedelta64(1, "h")
    width_hours = (check.time_bnds[:, 1] - check.time_bnds[:, 0]).values / np.timedelta64(1, "h")
    midpoints = check.time_bnds[:, 0].values + (check.time_bnds[:, 1].values - check.time_bnds[:, 0].values) / 2

    if not np.all(dt_hours == 3):
        raise ValueError(f"Unexpected time intervals: {np.unique(dt_hours)} h")
    if not np.all(width_hours == 3):
        raise ValueError(f"Unexpected time_bnds widths: {np.unique(width_hours)} h")
    if not np.array_equal(midpoints, check.time.values):
        raise ValueError("time is not the midpoint of time_bnds")

    if check.time.encoding.get("units") != "days since 1900-01-01":
        raise ValueError(f"Unexpected time units: {check.time.encoding.get('units')}")
    if check.time.encoding.get("calendar") != "gregorian":
        raise ValueError(f"Unexpected calendar: {check.time.encoding.get('calendar')}")

    tas_min = float(check[ACCESS_VARIABLE].min())
    tas_max = float(check[ACCESS_VARIABLE].max())
    print(f"\nRange [K]: {tas_min:.3f} to {tas_max:.3f}")

    source_first = tas.sel(time=check.time.values[0]).compute()
    output_first = check[ACCESS_VARIABLE].isel(time=0).load()
    max_abs_difference = float(np.abs(source_first - output_first).max())

    print("Maximum absolute difference after NetCDF round trip:", max_abs_difference)

    if max_abs_difference != 0.0:
        raise ValueError("NetCDF round trip changed tas values")

print("Output verification passed")

: 

: 